## <b> Textbasiertes Roulette mit Einsatz und Kontostand </b>

In [1]:
import random

def roulette():
    kontostand = 100
    rote_zahlen = [1, 3, 5, 7, 9, 12, 14, 16, 18, 19, 21, 23, 25, 27, 30, 32, 34, 36]

    print("--- Willkommen beim Python-Roulette! ---")
    print(f"Startguthaben: {kontostand} Euro")

    while kontostand > 0:
        print(f"\nDein aktueller Kontostand: {kontostand} Euro")
        
        # 1. Einsatz abfragen
        einsatz_input = input("Dein Einsatz (oder 'ende' zum Aufhören): ")
        if einsatz_input.lower() == 'ende':
            break
        
        einsatz = int(einsatz_input)
        if einsatz > kontostand:
            print("Du hast nicht genug Guthaben!")
            continue

        # 2. Worauf wird gesetzt?
        tipp = input("Worauf setzt du? (Zahl 0-36, 'rot' oder 'schwarz'): ").lower()

        # 3. Die Kugel rollt
        ergebnis_zahl = random.randint(0, 36)
        
        if ergebnis_zahl == 0:
            ergebnis_farbe = "grün"
        elif ergebnis_zahl in rote_zahlen:
            ergebnis_farbe = "rot"
        else:
            ergebnis_farbe = "schwarz"

        print(f"\nKugel landet auf: {ergebnis_zahl} ({ergebnis_farbe})")

        # 4. Gewinnprüfung
        gewonnen = False
        
        # Check: Hat der User auf die Farbe gesetzt?
        if tipp == ergebnis_farbe:
            gewonnen = True
            profit = einsatz  # Verdopplung (Einsatz zurück + Gewinn in Höhe des Einsatzes)
        
        # Check: Hat der User auf die Zahl gesetzt?
        elif tipp.isdigit() and int(tipp) == ergebnis_zahl:
            gewonnen = True
            profit = einsatz * 35 # 35-facher Gewinn
        
        else:
            gewonnen = False

        # 5. Kontostand aktualisieren
        if gewonnen:
            print(f"Gewonnen! Du erhältst {profit} Euro zusätzlich.")
            kontostand += profit
        else:
            print("Leider verloren.")
            kontostand -= einsatz

    print(f"Spiel beendet. Dein finaler Kontostand: {kontostand} Euro.")

# Spiel starten
roulette()

--- Willkommen beim Python-Roulette! ---
Startguthaben: 100 Euro

Dein aktueller Kontostand: 100 Euro


<class 'AttributeError'>: 'NoneType' object has no attribute 'lower'

## <b>Roulette-Animation mit Turtle</b>

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning) # gibt sonst eine nervige Fehlermeldung

import piplite
await piplite.install('ColabTurtlePlus')

from ColabTurtlePlus.Turtle import *
import random
import time

# --- Setup der Grafik ---
def initialisiere_rad():
    clearscreen()
    setup(500, 500)
    T = Turtle()
    T.speed(0)
    T.hideturtle()
    
    # Vereinfachtes Rad zeichnen
    rote_zahlen = [1, 3, 5, 7, 9, 12, 14, 16, 18, 19, 21, 23, 25, 27, 30, 32, 34, 36]
    
    for i in range(37):
        # Farbe bestimmen
        if i == 0:
            fill_color = "green"
        elif i in rote_zahlen:
            fill_color = "red"
        else:
            fill_color = "black"
            
        T.color("white", fill_color)
        T.begin_fill()
        
        # Segment zeichnen
        T.jumpto(0,0)
        T.setheading(i * (360/37))
        T.forward(150)
        T.left(90)
        T.circle(150, 360/37)
        T.goto(0,0)
        T.end_fill()
    
    return rote_zahlen

def kugel_animieren(ziel_zahl):
    kugel = Turtle()
    kugel.shape("circle")
    kugel.color("white")
    kugel.animationOff() # Schnelleres Bewegen für die Runden
    kugel.penup()
    
    # 2 schnelle Runden drehen
    for _ in range(2):
        for s in range(37):
            winkel = s * (360/37)
            kugel.setheading(winkel)
            kugel.goto(0,0)
            kugel.forward(130)
    
    kugel.animationOn()
    # Langsam auf der Zielzahl landen
    winkel_ziel = ziel_zahl * (360/37)
    kugel.setheading(winkel_ziel)
    kugel.goto(0,0)
    kugel.forward(130)
    return kugel

# --- Hauptlogik ---
rote_zahlen = initialisiere_rad()
kontostand = 100

print(f"Willkommen! Dein Kontostand: {kontostand}€")
tipp = input("Setze auf eine Zahl (0-36) oder eine Farbe (red/black): ")
einsatz = int(input(f"Dein Einsatz (max. {kontostand}): "))

# Kugel rollen
gewinn_zahl = random.randint(0, 36)
kugel_animieren(gewinn_zahl)

# Farbe der Gewinnzahl prüfen
gewinn_farbe = "green" if gewinn_zahl == 0 else ("red" if gewinn_zahl in rote_zahlen else "black")

print(f"Ergebnis: {gewinn_zahl} ({gewinn_farbe})")

# Gewinnprüfung (vereinfacht)
if tipp == str(gewinn_zahl) or tipp == gewinn_farbe:
    print("GEWONNEN!")
else:
    print("Leider verloren.")

## <b> Roulette(bzw. Glücksrad) - Animation mit ipycanvas </b>

In [ ]:
import piplite
await piplite.install('ipycanvas')

import numpy as np
import random
import time
import math
from ipycanvas import Canvas, hold_canvas

# --- Konfiguration ---
WIDTH, HEIGHT = 600, 400
RADIUS = 150
CENTER_X, CENTER_Y = 200, 200
ROTE_ZAHLEN = [1, 3, 5, 7, 9, 12, 14, 16, 18, 19, 21, 23, 25, 27, 30, 32, 34, 36]

# Canvas erstellen
canvas = Canvas(width=WIDTH, height=HEIGHT)
display(canvas)

def draw_wheel(rotation=0):
    """Zeichnet das Roulette-Rad mit der aktuellen Rotation."""
    canvas.clear()
    
    # 37 Segmente (0-36)
    segment_angle = (2 * math.pi) / 37
    
    for i in range(37):
        start_angle = i * segment_angle + rotation
        end_angle = (i + 1) * segment_angle + rotation
        
        # Farbe bestimmen
        if i == 0:
            color = 'green'
        elif i in ROTE_ZAHLEN:
            color = 'red'
        else:
            color = 'black'
            
        # Segment zeichnen
        canvas.fill_style = color
        canvas.begin_path()
        canvas.move_to(CENTER_X, CENTER_Y)
        canvas.arc(CENTER_X, CENTER_Y, RADIUS, start_angle, end_angle)
        canvas.close_path()
        canvas.fill()
        
        # Zahlentext zeichnen (optional)
        canvas.fill_style = 'white'
        canvas.font = '10px sans-serif'
        text_angle = start_angle + segment_angle / 2
        tx = CENTER_X + (RADIUS - 20) * math.cos(text_angle)
        ty = CENTER_Y + (RADIUS - 20) * math.sin(text_angle)
        canvas.fill_text(str(i), tx - 5, ty + 5)

    # Statischer Zeiger (Marker oben)
    canvas.stroke_style = 'gold'
    canvas.line_width = 3
    canvas.stroke_line(CENTER_X, CENTER_Y - RADIUS - 10, CENTER_X, CENTER_Y - RADIUS + 10)

def spin_animation(target_num):
    """Animiert das Rad bis zur Gewinnzahl."""
    # Berechne Winkel für die Zielzahl (0 oben)
    segment_angle = (2 * math.pi) / 37
    target_angle = -target_num * segment_angle - (math.pi / 2)
    
    current_rot = 0
    speed = 0.5
    deceleration = 0.98
    
    # Game Loop für die Animation
    while speed > 0.01:
        with hold_canvas(canvas):
            draw_wheel(current_rot)
            current_rot += speed
            speed *= deceleration
        time.sleep(0.02)
    
    # Finale Position setzen
    with hold_canvas(canvas):
        draw_wheel(target_angle)

# --- Start des Spiels ---
print("Wähle eine Zahl (0-36):")
tipp = int(input("> "))
gewinn_zahl = random.randint(0, 36)

print(f"Die Kugel rollt...")
spin_animation(gewinn_zahl)

print(f"\nERGEBNIS: {gewinn_zahl}")
if tipp == gewinn_zahl:
    print("Herzlichen Glückwunsch! Du hast gewonnen!")
else:
    print("Leider verloren. Probier es nochmal!")

In [ ]:
## <b>Blackjack mit Karten</b>

In [ ]:
import piplite
await piplite.install('ipycanvas')

import random
from ipycanvas import Canvas, hold_canvas
import time

# --- Konfiguration & Design ---
WIDTH, HEIGHT = 600, 400
CARD_W, CARD_H = 70, 100
canvas = Canvas(width=WIDTH, height=HEIGHT)
display(canvas)

def draw_card(x, y, label, color="black"):
    """Zeichnet eine einzelne Spielkarte mit abgerundeten Ecken manuell."""
    r = 10 # Radius der Ecken
    w, h = CARD_W, CARD_H
    
    # Schatten-Effekt (optional)
    canvas.fill_style = "white"
    
    # Pfad für abgerundetes Rechteck
    canvas.begin_path()
    canvas.move_to(x + r, y)
    canvas.arc_to(x + w, y, x + w, y + h, r)
    canvas.arc_to(x + w, y + h, x, y + h, r)
    canvas.arc_to(x, y + h, x, y, r)
    canvas.arc_to(x, y, x + w, y, r)
    canvas.close_path()
    
    # Karte füllen und Rahmen zeichnen
    canvas.fill()
    canvas.stroke_style = "black"
    canvas.line_width = 2
    canvas.stroke()
    
    # Text auf der Karte
    canvas.fill_style = color
    canvas.font = "bold 18px sans-serif"
    canvas.fill_text(label, x + 10, y + 25)

def update_ui(player_hand, dealer_hand, message=""):
    """Aktualisiert die gesamte Spielfläche."""
    with hold_canvas(canvas):
        canvas.clear()
        canvas.fill_style = "#2e7d32" # Casinotisch-Grün
        canvas.fill_rect(0, 0, WIDTH, HEIGHT)
        
        # Texte
        canvas.fill_style = "white"
        canvas.font = "20px sans-serif"
        canvas.fill_text(f"Dealer: {calculate_score(dealer_hand)}", 50, 50)
        canvas.fill_text(f"Spieler: {calculate_score(player_hand)}", 50, 230)
        canvas.fill_text(message, 50, 370)
        
        # Karten Dealer
        for i, card in enumerate(dealer_hand):
            draw_card(50 + i * 80, 70, card)
            
        # Karten Spieler
        for i, card in enumerate(player_hand):
            draw_card(50 + i * 80, 250, card)

# --- Spiellogik ---
def calculate_score(hand):
    score = 0
    aces = 0
    values = {"2":2, "3":3, "4":4, "5":5, "6":6, "7":7, "8":8, "9":9, "10":10, "J":10, "Q":10, "K":10, "A":11}
    for card in hand:
        score += values[card]
        if card == "A": aces += 1
    while score > 21 and aces:
        score -= 10
        aces -= 1
    return score

def play_blackjack():
    deck = ["2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K", "A"] * 4
    random.shuffle(deck)
    
    player_hand = [deck.pop(), deck.pop()]
    dealer_hand = [deck.pop()]
    
    game_over = False
    update_ui(player_hand, dealer_hand, "Willkommen! 'h' für Hit, 's' für Stand")
    
    # Spielerzug
    while calculate_score(player_hand) < 21:
        move = input("Zugeben (h) oder Halten (s)? ").lower()
        if move == 'h':
            player_hand.append(deck.pop())
            update_ui(player_hand, dealer_hand)
        else:
            break
            
    # Dealerzug & Auswertung
    score_p = calculate_score(player_hand)
    if score_p > 21:
        update_ui(player_hand, dealer_hand, "Überkauft! Dealer gewinnt.")
    else:
        while calculate_score(dealer_hand) < 17:
            dealer_hand.append(deck.pop())
            update_ui(player_hand, dealer_hand, "Dealer zieht...")
            time.sleep(0.5)
            
        score_d = calculate_score(dealer_hand)
        if score_d > 21 or score_p > score_d:
            update_ui(player_hand, dealer_hand, f"GEWONNEN! ({score_p} vs {score_d})")
        elif score_p < score_d:
            update_ui(player_hand, dealer_hand, f"VERLOREN! ({score_p} vs {score_d})")
        else:
            update_ui(player_hand, dealer_hand, "Unentschieden!")

play_blackjack()